# 📘 IMDB Sentiment Analysis with GRU
IT22923288: W G D D Gayashan
Course: Deep Learning Assignment  
Dataset: [IMDB Dataset of 50K Movie Reviews](https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews)

---

### Objective
Perform Sentiment Analysis on IMDB reviews using a GRU deep learning model.


In [ ]:
#  1. Imports

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
STOPWORDS = set(stopwords.words('english'))

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split


In [ ]:
#  2. Load dataset

df = pd.read_csv("../Data/IMDB Dataset.csv")
print("Shape:", df.shape)
df.head()


In [ ]:
#  3. Data cleaning

def clean_text(text):
    text = re.sub(r"<.*?>", " ", text)  # remove HTML tags
    text = re.sub(r"[^a-zA-Z']", " ", text)
    text = text.lower()
    tokens = text.split()
    tokens = [t for t in tokens if t not in STOPWORDS]
    return " ".join(tokens)

df["clean_review"] = df["review"].astype(str).apply(clean_text)
df["label"] = df["sentiment"].map({"positive":1, "negative":0})
df.head(3)


In [ ]:
#  Quick EDA (Exploratory Data Analysis)

plt.figure(figsize=(5,4))
sns.countplot(data=df, x="sentiment", palette="viridis")
plt.title("Class Distribution")
plt.show()

# review length distribution
df["review_length"] = df["clean_review"].apply(lambda x: len(x.split()))
plt.figure(figsize=(8,4))
sns.histplot(df["review_length"], bins=50, kde=True)
plt.title("Review Length Distribution")
plt.xlabel("Number of Words per Review")
plt.show()

print("Average review length:", df["review_length"].mean())


In [ ]:
# 5. Prepare data 

MAX_VOCAB = 20000
MAX_LEN = 200

X = df["clean_review"].values
y = df["label"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

tokenizer = Tokenizer(num_words=MAX_VOCAB, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding="post", truncating="post")
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LEN, padding="post", truncating="post")

print("Train shape:", X_train_pad.shape)
print("Test shape:", X_test_pad.shape)


In [ ]:
#  6. Save tokenizer & data  

import os, pickle
os.makedirs("../artifacts", exist_ok=True)

np.savez_compressed("../artifacts/data_from_notebook.npz",
                    X_train=X_train_pad, X_test=X_test_pad,
                    y_train=y_train, y_test=y_test)

with open("../artifacts/tokenizer_from_notebook.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

print("Data saved in artifacts/")


In [ ]:
#  7. Build GRU Model

from tensorflow.keras import layers, models, optimizers

def build_gru(max_vocab, embedding_dim=100, max_len=200, gru_units=128, dropout=0.5):
    model = models.Sequential([
        layers.Embedding(input_dim=max_vocab, output_dim=embedding_dim, input_length=max_len),
        layers.SpatialDropout1D(0.2),
        layers.GRU(gru_units, return_sequences=False),
        layers.Dense(64, activation='relu'),
        layers.Dropout(dropout),
        layers.Dense(1, activation='sigmoid')
    ])
    return model

model = build_gru(MAX_VOCAB, 100, MAX_LEN)
model.compile(optimizer=optimizers.Adam(1e-3),
              loss="binary_crossentropy",
              metrics=["accuracy"])

model.summary()


In [ ]:
#  8. Train GRU Model

from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

callbacks = [
    EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6)
]

history = model.fit(
    X_train_pad, y_train,
    validation_split=0.1,
    epochs=8,
    batch_size=128,
    callbacks=callbacks
)


In [ ]:
#  9. Plot training curves

plt.figure(figsize=(8,4))
plt.plot(history.history["accuracy"], label="Train Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.title("GRU Training Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

plt.figure(figsize=(8,4))
plt.plot(history.history["loss"], label="Train Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.title("GRU Training Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.show()


In [ ]:
#  10. Evaluate on Test Set

from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

preds = model.predict(X_test_pad).ravel()
pred_labels = (preds >= 0.5).astype(int)

print(classification_report(y_test, pred_labels, digits=4))
print("ROC AUC:", roc_auc_score(y_test, preds))

cm = confusion_matrix(y_test, pred_labels)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


In [ ]:
#  11. Save the trained model

model.save("../artifacts/models/gru_notebook.keras")
print("Model saved successfully ✅")


In [ ]:
# 12. Summary cell

print("✅ GRU Model Training Complete")
print("Test Accuracy:", np.mean(y_test == pred_labels))
print("ROC AUC:", roc_auc_score(y_test, preds))
print("Model and artifacts saved under '../artifacts/'")
